# 07 — Pruebas de funcionamiento y evidencia

Esta ejecución produce la evidencia de nuestra solución: 14 escenarios de texto y seis escenarios de voz con estado inicial reproducible; la comparación de voz con su transcripción correcta; nDCG@10, restricciones, errores ASR, calidad de voz y latencia por etapa. Los resultados se generan al ejecutar este notebook y no se reemplazan por cifras históricas.


In [ ]:
%pip -q install -r requirements-lab4-modular.txt

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
import importlib.metadata
import pandas as pd
from IPython.display import display
from lab4_asistente import config as C
from lab4_asistente.escenarios import ESCENARIOS, ejecutar
from lab4_asistente.evaluacion import (tabla_escenarios, resumen_escenarios,
    tabla_asr, tabla_latencias, tabla_calidad_tts, tabla_aciertos_fallos)
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz
from lab4_asistente.voz import Semejanza


## Seleccionar el perfil de voz

El menú se alimenta de las grabaciones disponibles. La siguiente celda carga los modelos y comprueba los audios de los seis escenarios.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

VOICES = C.voces_disponibles()
if not VOICES:
    raise FileNotFoundError("No se encontraron perfiles/grabaciones de voz.")
voice_selector = widgets.Dropdown(options=sorted(VOICES), description="Perfil de voz:")
display(voice_selector)
print("Elige la voz y ejecuta la siguiente celda para cargar modelos y escenarios.")


## Cargar modelos y preparar escenarios de voz

Qwen usa la GPU. Recuperación, Whisper y XTTS usan CPU durante esta evaluación para evitar que la carga simultánea exceda la VRAM local. Se miden las cargas iniciales por separado de la latencia de cada turno.


In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = voice_selector.value
VOZ = C.rutas_voz(VOICE_ID)
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID)
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
RUTA_AUDIOS = VOZ["audios"]
AUDIO_REQUERIDO = [(e["id"], t["audio"]) for e in ESCENARIOS if e["modo"] == "voz" for t in e["turnos"]]
FALTAN = [nombre for _, nombre in AUDIO_REQUERIDO if not (RUTA_AUDIOS / nombre).exists()]
print("Voz seleccionada:", VOICE_ID)
print("Dispositivos:", COMP["dispositivos"], "ASR/TTS:", cargas_voz["dispositivo_asr"], cargas_voz["dispositivo_tts"])
print("Cargas iniciales (por separado de latencias):", CARGAS_INICIALES)
print("Audios requeridos:", len(AUDIO_REQUERIDO), "| faltantes:", len(FALTAN))
if FALTAN:
    display(pd.DataFrame({"audios_faltantes": FALTAN}))


## Ejecutar los 14 escenarios de texto

In [ ]:
C.RUTA_ESCENARIOS.mkdir(parents=True, exist_ok=True)
RES_TEXTO = []
for e in [x for x in ESCENARIOS if x["modo"] == "texto"]:
    RES_TEXTO.append(ejecutar(e, SIS, "texto", C.RUTA_ESCENARIOS / "texto" / e["id"]))
print("Escenarios de texto ejecutados:", len(RES_TEXTO))


## Ejecutar los seis escenarios con audio

Las grabaciones se conservan junto con transcripciones y operaciones. Se ejecutan también con sus textos correctos para separar errores del ASR de errores del agente.

In [ ]:
assert not FALTAN, "Completa las grabaciones requeridas antes de ejecutar la evaluación de voz."
RES_VOZ, RES_VOZ_TEXTO = [], []
for e in [x for x in ESCENARIOS if x["modo"] == "voz"]:
    carpeta = C.RUTA_ESCENARIOS / "voz_audio" / e["id"]
    RES_VOZ.append(ejecutar(e, SIS, "voz", carpeta, RUTA_AUDIOS, ASR, TTS, sintetizar=True))
    carpeta_txt = C.RUTA_ESCENARIOS / "voz_transcripcion_correcta" / e["id"]
    RES_VOZ_TEXTO.append(ejecutar(e, SIS, "texto", carpeta_txt))
print("Escenarios de voz:", len(RES_VOZ), "| replay con transcripción correcta:", len(RES_VOZ_TEXTO))


COMPARACION_ASR = pd.DataFrame([
    {"escenario": audio["id"], "completado_con_audio": audio["completado"],
     "completado_con_transcripcion_correcta": texto["completado"],
     "diferencia_por_asr": bool(audio["completado"] != texto["completado"])}
    for audio, texto in zip(RES_VOZ, RES_VOZ_TEXTO)
])
print("Efecto de reemplazar ASR por la transcripción correcta:")
display(COMPARACION_ASR)


## Resumen, incumplimientos y métricas de ASR

In [ ]:
TODOS = RES_TEXTO + RES_VOZ
MATRIZ = tabla_escenarios(TODOS)
display(MATRIZ)
print("Resumen de escenarios:", resumen_escenarios(TODOS))

ASR_METRICAS = tabla_asr(RES_VOZ)
display(ASR_METRICAS)
if len(ASR_METRICAS):
    print("WER medio:", ASR_METRICAS.WER.mean())
    print("CER medio:", ASR_METRICAS.CER.mean())
    print("Errores de cantidades/importes/productos:",
          ASR_METRICAS.err_cantidades.sum(), ASR_METRICAS.err_importes.sum(), ASR_METRICAS.err_productos.sum())

COMPARACION_ASR = pd.DataFrame([
    {"escenario": audio["id"], "completado_con_audio": audio["completado"],
     "completado_con_transcripcion_correcta": texto["completado"],
     "diferencia_por_asr": bool(audio["completado"] != texto["completado"])}
    for audio, texto in zip(RES_VOZ, RES_VOZ_TEXTO)
])
print("Efecto de sustituir la transcripción ASR por el texto correcto:")
display(COMPARACION_ASR)


## Latencia por etapa y evaluación de voz sintetizada

In [ ]:
LATENCIAS = tabla_latencias(RES_VOZ)
display(LATENCIAS.groupby("etapa").segundos.agg(["mean", "median", "max"]).sort_values("mean", ascending=False))
print("Carga inicial separada de las latencias:", CARGAS_INICIALES)
SEMEJANZA = Semejanza(C.MODELO_SEMEJANZA, "cpu")
REF_VOZ = VOZ["referencia"]
TTS_METRICAS = tabla_calidad_tts(RES_VOZ, C.RUTA_ESCENARIOS / "voz_audio", ASR, SEMEJANZA, REF_VOZ)
display(TTS_METRICAS)


## Análisis de cinco aciertos y cinco fallos

In [ ]:
ANALISIS = tabla_aciertos_fallos(TODOS, limite=5)
print("Aciertos y fallos seleccionados desde esta ejecución. La causa se basa en el verificador y el registro real de herramientas.")
display(ANALISIS[["tipo", "escenario", "modo", "diagnostico_observado"]])
print("Evidencia detallada disponible en la columna evidencia y en los registros JSON por escenario.")


## Guardar resultados y configuración ejecutada

In [ ]:
C.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
MATRIZ.to_csv(C.RUTA_RESULTADOS / "metricas_escenarios.csv", index=False)
ASR_METRICAS.to_csv(C.RUTA_RESULTADOS / "metricas_asr.csv", index=False)
LATENCIAS.to_csv(C.RUTA_RESULTADOS / "latencias_por_turno.csv", index=False)
TTS_METRICAS.to_csv(C.RUTA_RESULTADOS / "metricas_tts_por_respuesta.csv", index=False)
paquetes = ["torch", "torchaudio", "transformers", "sentence-transformers", "TTS", "pandas", "pyarrow", "rank-bm25"]
versiones = {}
for nombre in paquetes:
    try:
        versiones[nombre] = importlib.metadata.version(nombre)
    except importlib.metadata.PackageNotFoundError:
        versiones[nombre] = "no instalado"
(C.RUTA_RESULTADOS / "versiones.json").write_text(pd.Series(versiones).to_json(indent=2), encoding="utf-8")
print("Resultados guardados en:", C.RUTA_RESULTADOS)

COMPARACION_ASR.to_csv(C.RUTA_RESULTADOS / "escenarios_audio_vs_transcripcion_correcta.csv", index=False)
ANALISIS.to_csv(C.RUTA_RESULTADOS / "analisis_cinco_aciertos_y_fallos.csv", index=False)
